In [1]:
import numpy as np
import matplotlib.pyplot as plt

from utils.problem_setup import TestProblemsSetup
from algorithms.cg_solvers import DynamicalLowRankPCG
from algorithms.rsvd_solvers import MatrixFreeRSVD
from utils.exact_forward_operator import ExactForwardOperatorFast, fast_get_weights

PROBLEMS = TestProblemsSetup(n=32).get_test_problems()



/home/elias/miniforge3/envs/fenics_env/lib/python3.9/site-packages/ufl/__init__.py:250: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
def get_exact_Hessian(lam):
    pb = PROBLEMS['I']
    exact = ExactForwardOperatorFast(pb['V_h'])

    K = exact.K
    M_d = exact.M_ds
    M = exact.M_dx
    W = np.diag(exact.get_weights())

    # H = K^T M_d K + lam² W^T M W
    H = K.T @ M_d @ K + lam**2 * W.T @ M @ W
    return K, H




In [3]:
for lam in [10, 1e-1, 1e-2, 1e-3, 1e-4]:
    print(f"lam = {lam}: lam^2 = {lam**2:.2e}")
    K, H = get_exact_Hessian(lam)
    cond_H = np.linalg.cond(H)
    cond_K = np.linalg.cond(K)
    print(f"cond(H)={cond_H:.2e}, cond(K)={cond_K:.2e}")

lam = 10: lam^2 = 1.00e+02


cond(H)=1.52e+03, cond(K)=7.06e+03
lam = 0.1: lam^2 = 1.00e-02
cond(H)=1.60e+03, cond(K)=7.06e+03
lam = 0.01: lam^2 = 1.00e-04
cond(H)=1.26e+05, cond(K)=7.06e+03
lam = 0.001: lam^2 = 1.00e-06
cond(H)=1.26e+07, cond(K)=7.06e+03
lam = 0.0001: lam^2 = 1.00e-08
cond(H)=1.26e+09, cond(K)=7.06e+03


In [16]:
pb = PROBLEMS['I']

rsvd = MatrixFreeRSVD(pb['V_h']); rsvd.solve(k=50)
solver = DynamicalLowRankPCG(rsvd)

X = solver.vec_to_matrix(pb['x'])

In [17]:
_, S, _ = np.linalg.svd(X)
S[:3]

array([5.00000000e+00, 4.46856148e-16, 4.46856148e-16])